# Replication Starter: Card (1995), Proximity to College and the Return to Schooling
## ECON 5200 · Midterm Replication Study · instrumental variables

Card, D. (1995). "Using Geographic Variation in College Proximity to Estimate the Return to Schooling." In
*Aspects of Labour Market Behaviour*, ed. Christofides, Grant and Swidinsky. University of Toronto Press.

> **AI rule for this project.** Sections 1 and 2 (positioning, direct replication) are **no AI**: not for
> the code, not for the prose. Sections 3 to 6 allow AI as a documented co-pilot under P.R.I.M.E., with the full
> transcript in `ai-appendix.md`. Every cell below is marked with the section it belongs to.

**How to use this notebook.** It is a starter, not the project. Copy it into your repository made from
[RyanPiao/econ5200-replication-template](https://github.com/RyanPiao/econ5200-replication-template), run it top to
bottom, and fill the cells marked **YOUR TURN**. Cells marked **GIVEN** run as they are; cells marked **CHECK** test
what you wrote and print a message, never the answer. A blank copy runs from top to bottom and prints no estimate:
the numbers appear once you have written Section 2.

In [ ]:
# GIVEN — setup: the IV package Colab does not have, then the libraries
%pip install linearmodels -q

import numpy as np
import pandas as pd
import statsmodels.api as sm
from linearmodels.iv import IV2SLS

print("Setup complete.")

## The data

`card.dta` is the NLS Young Men cohort as Card used it: men aged 14 to 24 in 1966, interviewed again in 1976. One row
per man. The variables the design uses:

| variable | what it is |
|---|---|
| `lwage` | log hourly wage, 1976 (the outcome) |
| `educ` | years of schooling completed by 1976 (the endogenous treatment) |
| `nearc4` | 1 if he grew up near a four-year college in 1966 (the instrument) |
| `nearc2` | 1 if near a two-year college in 1966 (a second instrument the paper discusses) |
| `exper`, `expersq` | potential experience and its square |
| `black`, `south`, `smsa` | race, and region and urban residence in 1976 |
| `reg661`–`reg669`, `smsa66` | region and urban residence in 1966 |

Read the paper's table notes before you choose controls: which of these does the specification you replicate
include, and which year's region variables?

In [ ]:
# GIVEN — Section 2 starts here: load the data
card = pd.read_stata("https://raw.githubusercontent.com/scunning1975/mixtape/master/card.dta")
print(f"{len(card):,} men, {card.shape[1]} columns")
card[["lwage", "educ", "nearc4", "nearc2", "exper", "black", "south", "smsa"]].describe().round(3)

## Section 2: direct replication — YOUR TURN, no AI

Write the two-stage least squares estimate of the return to schooling that the paper reports as its headline:
`lwage` on `educ`, instrumented by `nearc4`, with the controls the paper's table uses. Use `IV2SLS` from
`linearmodels` (`IV2SLS(dependent, exog, endog, instruments).fit(...)`; `sm.add_constant` builds `exog`).

Set two names, because Section 3 reuses them:

- `CONTROLS`: the list of control column names you include
- `iv_fit`: your fitted `IV2SLS` result
- `ols_fit`: the OLS regression of `lwage` on `educ` and the same controls (`sm.OLS`), the estimate IV is compared with

Then print the coefficient on `educ` and its standard error. Choose the covariance option the paper's standard
errors correspond to, and say which in your report.

In [ ]:
# YOUR TURN — Section 2, no AI: define CONTROLS, iv_fit and ols_fit, then print the estimate on educ
CONTROLS = None
iv_fit = None
ols_fit = None

In [ ]:
# GIVEN — Section 2: your replication table's first rows, OLS beside 2SLS, and the first-stage F
if iv_fit is None or ols_fit is None:
    print("Fill Section 2 first: iv_fit and ols_fit are not set yet.")
else:
    fs = sm.OLS(card["educ"], sm.add_constant(card[["nearc4"] + CONTROLS])).fit()
    print(f"{'':<8}{'return to a year':>18}{'SE':>9}")
    print(f"{'OLS':<8}{ols_fit.params['educ']:>18.4f}{ols_fit.bse['educ']:>9.4f}")
    print(f"{'2SLS':<8}{iv_fit.params['educ']:>18.4f}{iv_fit.std_errors['educ']:>9.4f}")
    print(f"first stage on nearc4: {fs.params['nearc4']:+.3f} years, F = {fs.tvalues['nearc4'] ** 2:.1f}")

### Your number, with its unit

`lwage` is a natural log, so the coefficient on `educ` is a change in the log wage per year of schooling. Times
100 it is in **log points** (0.10 is 10 log points), which is approximately the percent change in the wage;
100·(e^b − 1) is the exact percent. Both readings pass under "percent". It is not a percentage point: that unit belongs to
outcomes that are themselves proportions. Fill all five names:

- `PRINTED`: the coefficient exactly as your code prints it
- `IN_UNITS`: the same estimate in the unit you name in `UNIT`
- `UNIT`: one of `log points per year of schooling`, `percent per year of schooling`
- `PAPER_NUMBER`, `PAPER_WHERE`: the paper's own estimate, and its table, column and row, read from the paper

In [ ]:
# YOUR TURN — Section 2, no AI
PRINTED = None
IN_UNITS = None
UNIT = None
PAPER_NUMBER = None
PAPER_WHERE = None

In [ ]:
# CHECK — Section 2: your headline number, with its unit. Run it; you need not read it.
def check_headline(printed, in_units, unit, paper_number, paper_where, units_ok):
    """Is the unit right for this outcome, and is the in-unit number the printed one converted?"""
    if any(v is None for v in (printed, in_units, unit, paper_number, paper_where)):
        print("Not filled yet: set all five names in the cell above.")
        return False
    u = " ".join(str(unit).lower().split())
    if u not in units_ok:
        print(f"'{unit}' is not a unit this outcome is measured in. Allowed here: " + "; ".join(sorted(units_ok)) + ".")
        print("Ask what the outcome variable is: a log, a proportion (0 to 1), or a rate per 100,000.")
        return False
    wants = [f(float(printed)) for f in units_ok[u]]     # every accepted reading of this unit
    if not any(abs(float(in_units) - w) <= max(0.005, 0.02 * abs(w)) for w in wants):
        print(f"Your in-unit number does not follow from your printed estimate in '{unit}'.")
        print("Convert the printed coefficient again: what does one unit of the outcome mean here?")
        return False
    if not str(paper_where).strip() or len(str(paper_where).split()) < 2:
        print("Name where the paper's number is: table, column (or figure), and row.")
        return False
    print("Unit check passed. Now write the comparison in your report: your number, theirs, the gap, and its likely source.")
    return True

CARD_UNITS = {
    "log points per year of schooling": [lambda b: 100 * b],   # a log point is 100 x the change in the log
    "percent per year of schooling": [lambda b: 100 * b, lambda b: 100 * (np.exp(b) - 1)],   # approximation, or exact
}
ok = check_headline(PRINTED, IN_UNITS, UNIT, PAPER_NUMBER, PAPER_WHERE, CARD_UNITS)

## Section 3: robustness — GIVEN code, AI allowed from here

Each function takes your Section 2 choices, so the numbers below are about *your* specification. Run them, then
write what each one implies about the original claim. That third part is what is graded.

**1. The first stage, and how strong it is.** The first-stage F with one instrument is the instrument's t
statistic squared. The rule of thumb is F above 10; a valid conventional 5% t test needs about 104.7 (Lee,
McCrary, Moreira and Porter 2022). The heteroskedasticity-robust F is printed beside the classical one.

In [ ]:
# GIVEN — Section 3: the first stage for any controls, instrument and sample
def first_stage(controls, instrument="nearc4", data=None):
    data = card if data is None else data
    X = sm.add_constant(data[[instrument] + controls])
    classical = sm.OLS(data["educ"], X).fit()
    robust = sm.OLS(data["educ"], X).fit(cov_type="HC1")
    return {"coef": classical.params[instrument], "F": classical.tvalues[instrument] ** 2,
            "F_robust": robust.tvalues[instrument] ** 2, "n": int(classical.nobs)}


if CONTROLS is None:
    print("Fill Section 2 first: CONTROLS is not set yet.")
else:
    fs = first_stage(CONTROLS)
    print(f"First stage on nearc4: {fs['coef']:+.3f} years of schooling, "
          f"F = {fs['F']:.1f} (robust {fs['F_robust']:.1f}), n = {fs['n']:,}")

**2. A weak-instrument-robust interval.** Anderson–Rubin never divides by the first stage: for each candidate
return b0 it asks whether `lwage − b0·educ` is still related to the instrument, and keeps every b0 it cannot
reject. If the set runs into the edge of the grid, it is unbounded there, and the grid is not the answer.

In [ ]:
# GIVEN — Section 3: Anderson–Rubin 95% set over a grid of candidate returns
def anderson_rubin(controls, instrument="nearc4", grid=np.linspace(-0.2, 0.6, 801), data=None):
    data = card if data is None else data
    X = sm.add_constant(data[[instrument] + controls])
    keep = [b0 for b0 in grid
            if sm.OLS(data["lwage"] - b0 * data["educ"], X).fit(cov_type="HC1").pvalues[instrument] > 0.05]
    if not keep:
        return None
    edge = keep[0] == grid[0] or keep[-1] == grid[-1]
    return min(keep), max(keep), edge


if CONTROLS is None:
    print("Fill Section 2 first: CONTROLS is not set yet.")
else:
    ar = anderson_rubin(CONTROLS)
    if ar is None:
        print("No value on the grid survives: widen the grid.")
    else:
        lo, hi, edge = ar
        wald = iv_fit.conf_int().loc["educ"]
        print(f"Anderson-Rubin 95% set: [{lo:.3f}, {hi:.3f}]" + ("  (touches the grid edge: unbounded there)" if edge else ""))
        print(f"Your Section 2 Wald interval: [{wald.iloc[0]:.3f}, {wald.iloc[1]:.3f}]")

**3. Your own specification.** Change at least one choice the paper made and justify it: a different control set
(family background, the 1966 versus 1976 region variables), the other instrument (`nearc2`), or a subsample (for
example outside the South, or by race). `MY_SPEC` is a dictionary the code below understands:
`{"controls": [...], "instrument": "nearc4" or "nearc2", "sample": None or a pandas query string}`.

In [ ]:
# YOUR TURN — Section 3: your specification, and two sentences on why it is defensible
MY_SPEC = None
MY_SPEC_WHY = None

In [ ]:
# CHECK — Section 3: your own specification. Run it; you need not read it.
def check_my_spec(my_spec, baseline, why):
    if my_spec is None or why is None:
        print("Not filled yet: set MY_SPEC and MY_SPEC_WHY above.")
        return False
    if my_spec == baseline:
        print("MY_SPEC is the baseline. Change at least one choice, and say why that choice is defensible.")
        return False
    sentences = [s for s in str(why).replace("?", ".").split(".") if len(s.split()) >= 5]
    if len(sentences) < 2:
        print("MY_SPEC_WHY needs at least two sentences: what you changed, and why a careful author might have made that choice.")
        return False
    print("Specification accepted. Report it beside the baseline and say what the difference implies about the original claim.")
    return True

BASELINE = None if CONTROLS is None else {"controls": CONTROLS, "instrument": "nearc4", "sample": None}
if check_my_spec(MY_SPEC, BASELINE, MY_SPEC_WHY):
    d = card.query(MY_SPEC["sample"]) if MY_SPEC["sample"] else card
    z, c = MY_SPEC["instrument"], MY_SPEC["controls"]
    dropped = [x for x in c if d[x].nunique() < 2]           # a control that cannot vary in this sample
    c = [x for x in c if x not in dropped]
    if dropped:
        print("Dropped, constant in this sample:", ", ".join(dropped))
    fit = IV2SLS(d["lwage"], sm.add_constant(d[c]), d["educ"], d[z]).fit(cov_type=iv_fit.cov_type)
    fs = first_stage(c, z, d)
    print(f"Your specification: 2SLS {fit.params['educ']:.4f} (SE {fit.std_errors['educ']:.4f}), "
          f"first-stage F {fs['F']:.1f}, n = {fs['n']:,}")
    print(f"Section 2 baseline: 2SLS {iv_fit.params['educ']:.4f} (SE {iv_fit.std_errors['educ']:.4f})")

**4. Your other exercises.** Section 3 asks for at least four. Natural ones on this paper: the first stage and
2SLS in subsamples where the exclusion restriction is more or less plausible; `nearc2` and `nearc4` together, and
what an over-identification test can and cannot tell you; adding family background (`fatheduc`, `motheduc`,
`momdad14`), and what the rows lost to missing values do to the sample; a placebo outcome that proximity should not
move. Write them below this cell.

## Sections 4 to 6, in your repository

- **Section 4, does the finding hold?** Use your Section 3 results. "Holds in a narrower population", "holds at a
  smaller magnitude" and "holds under an assumption the authors did not state" are all answers.
- **Section 5, peer audit.** Assigned on the date the brief gives, never on your own paper; file the issue by the
  project deadline in the brief's Checkpoints list.
- **Section 6, your own repository.** Move the code you keep from this notebook into the template's layout, pin
  versions in `requirements.txt` (this notebook was built with pandas 2.2.2, numpy 2.0.2, statsmodels 0.14.6,
  linearmodels 7.1, rdrobust 2.0.0 and rddensity 3.0), and make the README name the one command that reproduces
  your headline table. Then clone your own repository into an empty folder and run that command.